In [ ]:
from google.colab import drive
drive.mount("/content/drive",force_remount=True)


Mounted at /content/drive


In [ ]:
import os

WORKDIR = "/content/drive/MyDrive/urdu_tts_clean"
os.makedirs(WORKDIR, exist_ok=True)

print("✅ WORKDIR:", WORKDIR)


✅ WORKDIR: /content/drive/MyDrive/urdu_tts_clean


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"atifkhan234","key":"078016e84cd9ca28719ddceeb9f0b73c"}'}



```
# This is formatted as code
```

**Install preprocessing deps (simple pip, no torch needed here)**

In [ ]:
!pip -q install kaggle
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle -v

Kaggle API 1.7.4.5


 ## Download & unzip the dataset**


```



In [ ]:
import os, glob, subprocess

# Where you want the dataset to live
DATA_ROOT = os.path.join(WORKDIR, "data")
UNZIP_DIR = os.path.join(DATA_ROOT, "urdu_dataset")
ZIP_PATH = os.path.join(DATA_ROOT, "urdu-dataset-20000.zip")

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(UNZIP_DIR, exist_ok=True)

# Detect whether dataset is already present (TSV + some audio)
tsv_exists = len(glob.glob(os.path.join(UNZIP_DIR, "**/*.tsv"), recursive=True)) > 0
wav_exists = len(glob.glob(os.path.join(UNZIP_DIR, "**/*.wav"), recursive=True)) > 100  # heuristic

print("tsv_exists:", tsv_exists)
print("wav_exists (>=100):", wav_exists)
print("UNZIP_DIR:", UNZIP_DIR)

if tsv_exists and wav_exists:
    print("✅ Dataset already present. Skipping download/unzip.")
else:
    # If zip doesn't exist, download it
    if not os.path.exists(ZIP_PATH):
        print("Downloading from Kaggle...")
        !kaggle datasets download -d muhammadahmedansari/urdu-dataset-20000 -p {DATA_ROOT}
    else:
        print("Zip already exists:", ZIP_PATH)

    print("Unzipping (this can take a while once)...")
    !unzip -q -o {ZIP_PATH} -d {UNZIP_DIR}

print("Done.")


tsv_exists: False
wav_exists (>=100): False
UNZIP_DIR: /content/drive/MyDrive/urdu_tts_clean/data/urdu_dataset
Dataset URL: https://www.kaggle.com/datasets/muhammadahmedansari/urdu-dataset-20000
License(s): other
100% 3.95G/3.95G [00:36<00:00, 181MB/s]
100% 3.95G/3.95G [00:36<00:00, 118MB/s]
Unzipping (this can take a while once)...
Done.


 ## Locate TSV + audio folder (robust)

In [ ]:
import os, glob

# Find TSV
tsvs = glob.glob(os.path.join(UNZIP_DIR, "**/*.tsv"), recursive=True)
print("TSVs found:", tsvs[:10])
assert len(tsvs) > 0, "No TSV found after unzip"
TSV_PATH = tsvs[0]
print("Using TSV:", TSV_PATH)

# Find audio folder
wav_files = glob.glob(os.path.join(UNZIP_DIR, "**/*.wav"), recursive=True)
mp3_files = glob.glob(os.path.join(UNZIP_DIR, "**/*.mp3"), recursive=True)

print("WAV count:", len(wav_files))
print("MP3 count:", len(mp3_files))

if len(wav_files) > 0:
    # choose the parent directory that contains most wav files
    # (simple heuristic: take directory of first wav)
    AUDIO_DIR = os.path.dirname(wav_files[0])
else:
    AUDIO_DIR = os.path.dirname(mp3_files[0])

print("AUDIO_DIR:", AUDIO_DIR)


TSVs found: ['/content/drive/MyDrive/urdu_tts_clean/data/urdu_dataset/final_main_dataset.tsv']
Using TSV: /content/drive/MyDrive/urdu_tts_clean/data/urdu_dataset/final_main_dataset.tsv
WAV count: 20000
MP3 count: 0
AUDIO_DIR: /content/drive/MyDrive/urdu_tts_clean/data/urdu_dataset/limited_wav_files/limited_wav_files




```
# This is formatted as code
```

**Install preprocessing deps (simple pip, no torch needed here)**

In [ ]:
!pip -q install pandas numpy tqdm soundfile librosa
!apt-get update -y
!apt-get install -y ffmpeg libsndfile1

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,361 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jam




```
# This is formatted as code
```

**Preprocess: load TSV, normalize text, resolve audio paths**

In [ ]:

import pandas as pd
import re
import unicodedata
import os
from tqdm import tqdm

# ----------------------------
# Load TSV
# ----------------------------
df = pd.read_csv(TSV_PATH, sep="\t")
print("Rows:", len(df))
print("Columns:", list(df.columns))

assert "client_id" in df.columns and "path" in df.columns and "sentence" in df.columns


# ----------------------------
# STRONG URDU NORMALIZATION
# ----------------------------
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")

def normalize_urdu(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text))

    # Remove zero-width characters
    text = re.sub(r"[\u200b-\u200f\u202a-\u202e]", "", text)

    # Remove Arabic diacritics
    text = ARABIC_DIACRITICS.sub("", text)

    # Normalize Arabic → Urdu forms
    text = text.replace("ي", "ی")
    text = text.replace("ك", "ک")
    text = text.replace("ى", "ی")   # ✅ FIX
    text = text.replace("أ", "ا")   # ✅ FIX
    text = text.replace("ۀ", "ۂ")

    # Normalize punctuation spacing
    text = text.replace("۔", "۔ ")
    text = text.replace("؟", "؟ ")
    text = text.replace("،", "، ")

    # Clean multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ----------------------------
# Clean Text Column
# ----------------------------
df["sentence"] = df["sentence"].astype(str).str.strip()
df = df[df["sentence"].str.len() > 0].copy()

df["text"] = df["sentence"].apply(normalize_urdu)


# ----------------------------
# Remove Strange Characters (Quality Filter)
# ----------------------------
allowed_pattern = re.compile(r"^[\u0600-\u06FF0-9\s\.\,\؟\!\-]+$")

before = len(df)
df = df[df["text"].apply(lambda x: bool(allowed_pattern.match(x)))]
print("Removed weird text rows:", before - len(df))


# ----------------------------
# Text Length Filter (Important for TTS)
# ----------------------------
df["text_len"] = df["text"].apply(len)

before_len = len(df)
df = df[(df["text_len"] >= 10) & (df["text_len"] <= 250)]
print("Text length filter removed:", before_len - len(df))


# ----------------------------
# Resolve Audio Paths
# ----------------------------
def resolve_audio(p: str) -> str:
    p = str(p)
    cand = os.path.join(AUDIO_DIR, p)

    if os.path.exists(cand):
        return cand

    base, ext = os.path.splitext(p)

    # Common mismatch: TSV lists mp3 but audio is wav
    for alt in [".wav", ".mp3"]:
        cand2 = os.path.join(AUDIO_DIR, base + alt)
        if os.path.exists(cand2):
            return cand2

    return cand  # will be filtered out


df["abs_path"] = df["path"].apply(resolve_audio)

exists = df["abs_path"].apply(os.path.exists)
print("Audio exists %:", round(exists.mean()*100, 2), "Missing:", int((~exists).sum()))

df = df[exists].copy()


# ----------------------------
# Optional Vote Filter
# ----------------------------
if "up_votes" in df.columns and "down_votes" in df.columns:
    df["up_votes"] = pd.to_numeric(df["up_votes"], errors="coerce").fillna(0)
    df["down_votes"] = pd.to_numeric(df["down_votes"], errors="coerce").fillna(0)
    df = df[df["down_votes"] <= df["up_votes"]].copy()


print("After filters:", len(df))

# Preview final cleaned dataset
df[["client_id", "path", "abs_path", "text"]].head()

Rows: 20000
Columns: ['client_id', 'path', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']
Removed weird text rows: 260
Text length filter removed: 375
Audio exists %: 100.0 Missing: 0
After filters: 19365


,client_id,path,abs_path,text
0,e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...,common_voice_ur_31771683.mp3,/content/drive/MyDrive/urdu_tts_clean/data/urd...,کبھی کبھار ہی خیالی پلاو بناتا ہوں
1,e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...,common_voice_ur_31771684.mp3,/content/drive/MyDrive/urdu_tts_clean/data/urd...,اور پھر ممکن ہے کہ پاکستان بھی ہو
2,e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...,common_voice_ur_31771685.mp3,/content/drive/MyDrive/urdu_tts_clean/data/urd...,یہ فیصلہ بھی گزشتہ دو سال میں
3,e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...,common_voice_ur_31771730.mp3,/content/drive/MyDrive/urdu_tts_clean/data/urd...,ان کے بلے بازوں کے سامنے ہو گا
4,e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...,common_voice_ur_31771732.mp3,/content/drive/MyDrive/urdu_tts_clean/data/urd...,آبی جانور میں بطخ بگلا اور دوسرا آبی پرندہ شام...


In [ ]:
# ----------------------------
# Character Coverage Check
# ----------------------------

all_text = " ".join(df["text"].tolist())
unique_chars = sorted(set(all_text))

print("Total unique characters:", len(unique_chars))
print("Characters:\n", "".join(unique_chars))

Total unique characters: 55
Characters:
  !,-.،؛؟ءآؤئابتثجحخدذرزسشصضطظعغفقلمنهوٹپچڈڑژکگںھہۂۃیےۓ۔


**Standardize audio to 16k mono wav (into WORKDIR)**

This makes training consistent across systems.

In [ ]:

import os, subprocess
from tqdm import tqdm
import soundfile as sf

BUNDLE_DIR = os.path.join(WORKDIR, "urdu_tts_bundle")
WAV_OUT = os.path.join(BUNDLE_DIR, "wavs")
os.makedirs(WAV_OUT, exist_ok=True)


# ----------------------------
# Robust Audio Conversion
# ----------------------------
def to_16k_mono_wav(in_path, out_path, sr=16000):
    cmd = [
        "ffmpeg", "-y",
        "-i", in_path,
        "-ac", "1",
        "-ar", str(sr),
        "-vn",
        "-acodec", "pcm_s16le",
        out_path
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    return os.path.exists(out_path)


# ----------------------------
# Convert Audio Files
# ----------------------------
wav_paths = []
failed = 0

for p in tqdm(df["abs_path"].tolist(), desc="Audio -> 16k mono wav"):

    base = os.path.splitext(os.path.basename(p))[0]
    out = os.path.join(WAV_OUT, base + ".wav")

    # Skip if already converted and valid
    if os.path.exists(out):
        try:
            info = sf.info(out)
            if info.frames > 1000:  # not empty
                wav_paths.append(out)
                continue
        except:
            pass

    # Convert fresh
    success = to_16k_mono_wav(p, out)

    if success:
        wav_paths.append(out)
    else:
        failed += 1
        wav_paths.append(None)


# Remove failed conversions
df["wav"] = wav_paths
df = df[df["wav"].notna()].copy()

print("Prepared wavs:", len(df))
print("FFmpeg failed files:", failed)

Audio -> 16k mono wav: 100%|██████████| 19365/19365 [42:58<00:00,  7.51it/s]


Prepared wavs: 19365
FFmpeg failed files: 0


**Duration filter (1–12s) + speaker-disjoint split**

In [ ]:

import soundfile as sf
import numpy as np
from tqdm import tqdm

# ----------------------------
# Duration Calculation
# ----------------------------
def get_dur(path):
    try:
        info = sf.info(path)
        return info.frames / info.samplerate
    except:
        return None


df["dur"] = [get_dur(p) for p in tqdm(df["wav"], desc="Durations")]
df = df[df["dur"].notna()].copy()

before = len(df)

# 🔥 Slightly stricter duration range (better for TTS stability)
df = df[(df["dur"] >= 1.0) & (df["dur"] <= 10.0)].copy()

print("Duration filter:", before, "->", len(df))
print("Average duration:", round(df["dur"].mean(), 3))
print("Min duration:", round(df["dur"].min(), 3))
print("Max duration:", round(df["dur"].max(), 3))


# ----------------------------
# Speaker Split (Robust)
# ----------------------------
rng = np.random.default_rng(42)

# Remove speakers with too few samples (important for stable validation)
speaker_counts = df["client_id"].astype(str).value_counts()
valid_speakers = speaker_counts[speaker_counts >= 5].index

df = df[df["client_id"].astype(str).isin(valid_speakers)].copy()

speakers = df["client_id"].astype(str).unique().tolist()
rng.shuffle(speakers)

n = len(speakers)

# 🔥 Better split: 90 / 5 / 5
train_spk = set(speakers[:int(0.90*n)])
val_spk   = set(speakers[int(0.90*n):int(0.95*n)])
test_spk  = set(speakers[int(0.95*n):])

def split_of(spk):
    if spk in train_spk:
        return "train"
    if spk in val_spk:
        return "val"
    return "test"

df["split"] = df["client_id"].astype(str).apply(split_of)

print("\nSplit distribution:")
print(df["split"].value_counts())

print("\nSpeakers count:")
print("Train speakers:", len(train_spk))
print("Val speakers:", len(val_spk))
print("Test speakers:", len(test_spk))

Durations: 100%|██████████| 19365/19365 [01:17<00:00, 251.33it/s]


Duration filter: 19365 -> 19304
Average duration: 4.206
Min duration: 1.008
Max duration: 9.936

Split distribution:
split
train    17121
val       1335
test       836
Name: count, dtype: int64

Speakers count:
Train speakers: 149
Val speakers: 8
Test speakers: 9


**Export manifests + meta (bundle is now ready)**

In [ ]:


import json, shutil, os

MANIFEST_DIR = os.path.join(BUNDLE_DIR, "manifests")
META_DIR = os.path.join(BUNDLE_DIR, "meta")
os.makedirs(MANIFEST_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)


# ----------------------------
# Write JSONL Manifest (Portable)
# ----------------------------
def write_jsonl(split):
    out = os.path.join(MANIFEST_DIR, f"{split}.jsonl")
    subset = df[df["split"] == split].copy()

    # Safety: remove empty text rows
    subset = subset[subset["text"].str.len() > 0]

    with open(out, "w", encoding="utf-8") as f:
        for _, r in subset.iterrows():

            # ✅ Save RELATIVE audio path (portable)
            rel_audio = os.path.relpath(r["wav"], BUNDLE_DIR)

            ex = {
                "audio": rel_audio,
                "text": r["text"],
                "speaker": str(r["client_id"])
            }

            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    return out, len(subset)


# Write splits
for s in ["train", "val", "test"]:
    p, k = write_jsonl(s)
    print(s, k, p)


# ----------------------------
# Save Speaker List
# ----------------------------
with open(os.path.join(META_DIR, "speakers.txt"), "w", encoding="utf-8") as f:
    for spk in sorted(df["client_id"].astype(str).unique()):
        f.write(spk + "\n")


# ----------------------------
# Example Reference WAV
# ----------------------------
example = df.sample(1, random_state=42).iloc[0]["wav"]
shutil.copy(example, os.path.join(META_DIR, "example_ref.wav"))


# ----------------------------
# Save Dataset Stats
# ----------------------------
stats = {
    "rows_total": int(len(df)),
    "speakers_total": int(df["client_id"].astype(str).nunique()),
    "avg_dur_sec": float(df["dur"].mean()),
    "splits": df["split"].value_counts().to_dict()
}

with open(os.path.join(META_DIR, "stats.json"), "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)


print("\n✅ Bundle ready at:", BUNDLE_DIR)
stats

train 17121 /content/drive/MyDrive/urdu_tts_clean/urdu_tts_bundle/manifests/train.jsonl
val 1335 /content/drive/MyDrive/urdu_tts_clean/urdu_tts_bundle/manifests/val.jsonl
test 836 /content/drive/MyDrive/urdu_tts_clean/urdu_tts_bundle/manifests/test.jsonl

✅ Bundle ready at: /content/drive/MyDrive/urdu_tts_clean/urdu_tts_bundle


{'rows_total': 19292,
 'speakers_total': 166,
 'avg_dur_sec': 4.2061387103462575,
 'splits': {'train': 17121, 'val': 1335, 'test': 836}}

**Training phase (reproducible env)**

Install micromamba + create Python 3.10 env

In [3]:
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj
!./bin/micromamba create -y -n tts python=3.10

info/files
info/has_prefix
info/index.json
info/test/run_test.sh
info/hash_input.json
info/recipe/C_ARES_LICENSE.txt
info/licenses/C_ARES_LICENSE.txt
info/recipe/conda_build_config.yaml
info/licenses/REPROC_LICENSE.txt
info/recipe/REPROC_LICENSE.txt
info/licenses/NLOHMANN_JSON_LICENSE.txt
info/recipe/NLOHMANN_JSON_LICENSE.txt
info/recipe/CURL_LICENSE.txt
info/licenses/CURL_LICENSE.txt
info/recipe/CPP_FILESYSTEM_LICENSE.txt
info/recipe/ZLIB_LICENSE.txt
info/licenses/ZLIB_LICENSE.txt
info/licenses/LIBNGHTTP2_LICENSE.txt
info/recipe/LIBNGHTTP2_LICENSE.txt
info/paths.json
info/recipe/LIBLZ4_LICENSE.txt
info/licenses/LIBLZ4_LICENSE.txt
info/recipe/SPDLOG_LICENSE.txt
info/licenses/SPDLOG_LICENSE.txt
info/licenses/FMT_LICENSE.txt
info/recipe/FMT_LICENSE.txt
info/licenses/LIBSOLV_LICENSE.txt
info/recipe/LIBSOLV_LICENSE.txt
info/licenses/mamba/LICENSE
info/recipe/build.sh
info/licenses/ZSTD_LICENSE.txt
info/recipe/TERMCOLOR_CPP_LICENSE.txt
info/recipe/ZSTD_LICENSE.txt
info/recipe/recipe-scripts

**Activate env + install pinned deps + ffmpeg**

In [4]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

apt-get update -y
apt-get install -y ffmpeg libsndfile1 git

python -m pip install --upgrade pip

# Torch for Py3.10 (works well on Colab GPU)
pip install torch==2.1.2 torchaudio==2.1.2 torchvision==0.16.2

pip install \
  numpy==1.26.4 pandas==2.1.4 scipy==1.11.4 \
  librosa==0.10.1 soundfile==0.12.1 tqdm==4.66.1 pyarrow==14.0.2 webrtcvad==2.0.10 \
  transformers==4.41.2 accelerate==0.31.0 sentencepiece==0.2.0 einops==0.8.0 \
  omegaconf==2.3.0 hydra-core==1.3.2

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,475 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,954 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,842 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,955 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/u

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
%%bash
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts
python -c "import sys, torch; print(sys.version); print(torch.__version__); print('cuda', torch.cuda.is_available())"

3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]
2.1.2+cu121
cuda True


**Clone repos for inference + fine-tuning**

In [ ]:
%%bash
mkdir -p /content/drive/MyDrive/urdu_tts_clean/repos


In [ ]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

cd /content/drive/MyDrive/urdu_tts_clean/repos

# Remove old copies if exist
rm -rf chatterbox-TTS chatterbox-finetuning

# Clone repos
git clone https://github.com/byCyrus/chatterbox-TTS.git
git clone https://github.com/gokhaneraslan/chatterbox-finetuning.git

echo "✅ Repos cloned successfully inside Drive!"


✅ Repos cloned successfully inside Drive!


Cloning into 'chatterbox-TTS'...
Cloning into 'chatterbox-finetuning'...
Updating files: 100% (72/72), done.


**Install fine-tuning repo deps safely (without overwriting torch)**

In [ ]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

REPO_DIR="/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning"

cd "$REPO_DIR"

echo "---- repo requirements.txt (first 200 lines) ----"
sed -n '1,200p' requirements.txt || true

# Install requirements but skip torch pins if they exist
python - << 'PY'
import pathlib, subprocess, sys

repo = pathlib.Path("/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning")
req = repo / "requirements.txt"

if not req.exists():
    print("No requirements.txt found; skipping")
    sys.exit(0)

lines = [
    l.strip() for l in req.read_text().splitlines()
    if l.strip() and not l.strip().startswith("#")
]

# Skip torch packages (we install them separately)
skip_prefixes = ("torch", "torchaudio", "torchvision")

filtered = [
    l for l in lines
    if not any(l.lower().startswith(p) for p in skip_prefixes)
]

print("✅ Installing filtered requirements count:", len(filtered))

if filtered:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *filtered])
PY


---- repo requirements.txt (first 200 lines) ----
peft==0.17.1
torch==2.6.0
torchaudio==2.6.0
torchvision==0.21.0
chatterbox-tts==0.1.2
silero-vad==6.2.0
librosa==0.11.0
soundfile==0.13.1
num2words
ffmpeg-python
tqdm
pandas
safetensors
tensorboard
omegaconf
hf_transfer
pyloudnorm
gdownCollecting peft==0.17.1
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 25.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 45.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 34.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 38.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.16.2 requires torch==2.1.2, but you have torch 2.6.0 which is incompatible.


**Configure training (manifests + turbo)**

This repo’s config entrypoints can vary, so here’s the robust approach:

- print the README quickstart

- locate a config file / train script

- set manifest paths and output dir

In [ ]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

REPO_DIR="/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning"

cd "$REPO_DIR"

echo "---- README (first 220 lines) ----"
sed -n '1,220p' README.md || true

echo "---- files ----"
ls -la

echo "---- find train entrypoints ----"
find . -maxdepth 2 -type f \( -name "train*.py" -o -name "*config*.py" -o -name "*.yaml" -o -name "*.yml" \) | sed -n '1,200p'


---- README (first 220 lines) ----
# Chatterbox: Fine-Tuning Inference Kit for TTS & Turbo 🎙️

> ## 🚀 **NEW: Chatterbox Turbo Support ADDED!** 🚀
>
> This repository now fully supports the fine-tuning of the **Chatterbox Turbo** model!
>
> *   **What is it?** A faster, GPT-2 based architecture with a strong English foundation.
> *   **Smart Multi-Language Support:** The setup script **automatically merges** Turbo's large English vocabulary with our custom 23-language grapheme set.
> *   **The Result:** Get the speed and quality of Turbo while seamlessly fine-tuning on new languages like Turkish, French, Spanish, and more.
>
> Read the **Standart vs. Turbo Modes** section below for details!


---

A modular infrastructure for **fine-tuning** both **Chatterbox TTS (Standart)** and **Chatterbox Turbo** models with your own dataset and generating high-quality speech synthesis.

This kit is specially designed to support **new languages** by intelligently extending the model's vocabulary for 

In [ ]:
import os, pandas as pd, shutil

# ----------------------------
# Paths (CLEAN PROJECT)
# ----------------------------
BUNDLE_DIR = "/content/drive/MyDrive/urdu_tts_clean/urdu_tts_bundle"
MANIFEST_TRAIN = os.path.join(BUNDLE_DIR, "manifests", "train.jsonl")

# Repo path inside Drive (clean)
REPO_ROOT = "/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning"

# Target dataset folder inside repo
TARGET_ROOT = os.path.join(REPO_ROOT, "MyTTSDataset")
TARGET_WAVS = os.path.join(TARGET_ROOT, "wavs")
os.makedirs(TARGET_WAVS, exist_ok=True)

# ----------------------------
# Load Train Manifest
# ----------------------------
df = pd.read_json(MANIFEST_TRAIN, lines=True)
print("Train rows:", len(df))

# ----------------------------
# Copy WAVs + Build metadata.csv
# ----------------------------
meta_path = os.path.join(TARGET_ROOT, "metadata.csv")

rows = []
copied = 0

for _, r in df.iterrows():

    # Manifest audio is relative → convert to absolute
    rel_audio = r["audio"]
    src = os.path.join(BUNDLE_DIR, rel_audio)

    # Stable filename
    base = os.path.splitext(os.path.basename(src))[0]
    dst = os.path.join(TARGET_WAVS, base + ".wav")

    if not os.path.exists(dst):
        shutil.copy(src, dst)
        copied += 1

    # LJSpeech format: file_id|text|normalized_text
    file_id = base
    text = str(r["text"])

    rows.append((file_id, text, text))

# Save metadata.csv
pd.DataFrame(rows).to_csv(
    meta_path,
    sep="|",
    header=False,
    index=False,
    encoding="utf-8"
)

print("✅ Wrote:", meta_path)
print("✅ Copied wavs:", copied)
print("✅ Total wavs present:", len(os.listdir(TARGET_WAVS)))


Train rows: 17121
✅ Wrote: /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning/MyTTSDataset/metadata.csv
✅ Copied wavs: 13071
✅ Total wavs present: 13364


In [ ]:
%%bash
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

# Go to repo inside CLEAN Drive folder
sed -n '1,220p' /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning/src/config.py


from dataclasses import dataclass

@dataclass
class TrainConfig:
    # --- Paths ---
    # Directory where setup.py downloaded the files
    model_dir: str = "./pretrained_models"
    
    # Path to your metadata CSV (Format: ID|RawText|NormText)
    csv_path: str = "./MyTTSDataset/metadata.csv"
    metadata_path: str = "./metadata.json"
    
    # Directory containing WAV files
    wav_dir: str = "./MyTTSDataset/wavs"
    #wav_dir: str = "./FileBasedDataset"
    
    preprocessed_dir = "./MyTTSDataset/preprocess"
    #preprocessed_dir = "./FileBasedDataset/preprocess"
    
    # Output directory for the finetuned model
    output_dir: str = "./chatterbox_output"
    
    is_inference = False
    inference_prompt_path: str = "./speaker_reference/2.wav"
    inference_test_text: str = "Merhaba, sesimi geliştirmem oldukça uzun zaman aldı ve şimdi sahip olduğuma göre, sessiz kalmayacağım."


    ljspeech = True # Set True if the dataset format is ljspeech, and False if it's file-based.
   

In [ ]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

# Go to repo inside Drive
cd /content/drive/MyDrive/urdu_tts/repos/chatterbox-finetuning

# Clean pretrained models for safe setup
rm -rf pretrained_models

# Run setup
python setup.py

--- Chatterbox Pretrained Model Setup ---

Creating directory: pretrained_models
Mode: CHATTERBOX-TTS (Checking 5 files)
Downloading: ve.safetensors...
Download complete: pretrained_models/ve.safetensors

Downloading: t3_cfg.safetensors...
Download complete: pretrained_models/t3_cfg.safetensors

Downloading: s3gen.safetensors...
Download complete: pretrained_models/s3gen.safetensors

Downloading: conds.pt...
Download complete: pretrained_models/conds.pt

Downloading: tokenizer.json...
Download complete: pretrained_models/tokenizer.json


INSTALLATION COMPLETE (CHATTERBOX-TTS MOD)
All models are set up in 'pretrained_models/' folder.
Note: 'grapheme_mtl_merged_expanded_v1.json' was saved as 'tokenizer.json' for the new vocabulary.


ve.safetensors: 100%|██████████| 5.43M/5.43M [00:00<00:00, 56.7MiB/s]
t3_cfg.safetensors: 100%|██████████| 1.98G/1.98G [00:24<00:00, 86.5MiB/s]
s3gen.safetensors: 100%|██████████| 0.98G/0.98G [00:13<00:00, 77.7MiB/s]
conds.pt: 100%|██████████| 105k/105k [00:00<00:00, 3.61MiB/s]
tokenizer.json: 68.4kiB [00:00, 35.2MiB/s]


In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning"

echo "Checking inference.py..."
grep -n "VOCAB" $REPO/inference.py || true

echo "Checking config.py..."
grep -n "new_vocab_size" $REPO/src/config.py || true


Checking inference.py...
Checking config.py...
38:    new_vocab_size: int = 52260 if is_turbo else 2454 


In [ ]:
%%bash
REPO="/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning"

ls -la $REPO/src | grep preprocess || true


-rw------- 1 root root 4983 Feb 14 18:02 preprocess_file_based.py
-rw------- 1 root root 5140 Feb 14 18:02 preprocess_json.py
-rw------- 1 root root 4402 Feb 14 18:02 preprocess_ljspeech.py


In [ ]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

REPO="/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning"

cd $REPO

# Clean pretrained models (safe for fresh setup)
rm -rf pretrained_models

python setup.py


--- Chatterbox Pretrained Model Setup ---

Creating directory: pretrained_models
Mode: CHATTERBOX-TTS (Checking 5 files)
Downloading: ve.safetensors...
Download complete: pretrained_models/ve.safetensors

Downloading: t3_cfg.safetensors...
Download complete: pretrained_models/t3_cfg.safetensors

Downloading: s3gen.safetensors...
Download complete: pretrained_models/s3gen.safetensors

Downloading: conds.pt...
Download complete: pretrained_models/conds.pt

Downloading: tokenizer.json...
Download complete: pretrained_models/tokenizer.json


INSTALLATION COMPLETE (CHATTERBOX-TTS MOD)
All models are set up in 'pretrained_models/' folder.
Note: 'grapheme_mtl_merged_expanded_v1.json' was saved as 'tokenizer.json' for the new vocabulary.


ve.safetensors: 100%|██████████| 5.43M/5.43M [00:00<00:00, 43.3MiB/s]
t3_cfg.safetensors: 100%|██████████| 1.98G/1.98G [00:39<00:00, 54.3MiB/s]
s3gen.safetensors: 100%|██████████| 0.98G/0.98G [00:13<00:00, 75.6MiB/s]
conds.pt: 100%|██████████| 105k/105k [00:00<00:00, 2.06MiB/s]
tokenizer.json: 68.4kiB [00:00, 34.7MiB/s]


In [6]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

pip install perth


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for perth: filename=perth-1.0.0-py3-none-any.whl size=1932 sha256=302720612fc1a260b1d75734f334d90ddb1dfd55368ae3b737307d4b71219063
  Stored in directory: /root/.cache/pip/wheels/7f/4d/fa/44ff06e9c8d0b1317f16a6cb761778b8db1fb705caf7b218ea
Successfully built perth


In [ ]:
import os
import csv
import pandas as pd

REPO_ROOT = "/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning"
TARGET_ROOT = os.path.join(REPO_ROOT, "MyTTSDataset")
BUNDLE_DIR = "/content/drive/MyDrive/urdu_tts_clean/urdu_tts_bundle"

MANIFEST = os.path.join(BUNDLE_DIR, "manifests", "train.jsonl")
META_FILE = os.path.join(TARGET_ROOT, "metadata.csv")

df = pd.read_json(MANIFEST, lines=True)
print("Rows in manifest:", len(df))

with open(META_FILE, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="|")
    for _, r in df.iterrows():
        base = os.path.splitext(os.path.basename(r["audio"]))[0]
        text = str(r["text"]).replace("|", "")
        writer.writerow([base, text, text])

print("✅ metadata.csv recreated")
print("Lines written:", sum(1 for _ in open(META_FILE, encoding="utf-8")))


Rows in manifest: 17121
✅ metadata.csv recreated
Lines written: 17121


In [ ]:
%%bash
head -3 /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning/MyTTSDataset/metadata.csv
wc -l /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning/MyTTSDataset/metadata.csv


common_voice_ur_31771683|کبھی کبھار ہی خیالی پلاو بناتا ہوں|کبھی کبھار ہی خیالی پلاو بناتا ہوں
common_voice_ur_31771684|اور پھر ممکن ہے کہ پاکستان بھی ہو|اور پھر ممکن ہے کہ پاکستان بھی ہو
common_voice_ur_31771685|یہ فیصلہ بھی گزشتہ دو سال میں|یہ فیصلہ بھی گزشتہ دو سال میں
17121 /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning/MyTTSDataset/metadata.csv


In [ ]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

cd /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning

# Clean old preprocess folder
rm -rf MyTTSDataset/preprocess
mkdir -p MyTTSDataset/preprocess

# Run preprocessing
python -m src.preprocess_ljspeech


2026-02-14 18:44:31 - __main__ - INFO - <class 'src.chatterbox_.tts.ChatterboxTTS'> engine starting...
2026-02-14 18:44:44 - __main__ - INFO - Processing dataset... Total: 17121
2026-02-14 19:01:39 - __main__ - ERROR - Error (common_voice_ur_31918692.wav): Failed to open the input "MyTTSDataset/wavs/common_voice_ur_31918692.wav" (Invalid data found when processing input).
Exception raised from get_input_format_context at /__w/audio/audio/pytorch/audio/src/libtorio/ffmpeg/stream_reader/stream_reader.cpp:42 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x96 (0x7ad54296c1b6 in /root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/torch/lib/libc10.so)
frame #1: c10::detail::torchCheckFail(char const*, char const*, unsigned int, std::string const&) + 0x64 (0x7ad542915a76 in /root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/torch/lib/libc10.so)
frame #2: <unknown function> + 0x42034 (0x7ad4b8aea034 in /root/.local/share/mamba/

/root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
INFO:__main__:Processing dataset... Total: 17121
 14%|█▍        | 2448/17121 [16:54<2:19:15,  1.76

In [ ]:
 %%bash
ls -lh /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning/MyTTSDataset/preprocess | head


total 61M
-rw------- 1 root root 4.5K Feb 14 19:27 common_voice_ur_26562732.pt
-rw------- 1 root root 5.1K Feb 14 19:27 common_voice_ur_26562733.pt
-rw------- 1 root root 4.3K Feb 14 19:27 common_voice_ur_26562734.pt
-rw------- 1 root root 4.3K Feb 14 19:27 common_voice_ur_26562735.pt
-rw------- 1 root root 5.4K Feb 14 19:27 common_voice_ur_26562736.pt
-rw------- 1 root root 4.6K Feb 14 19:27 common_voice_ur_26562738.pt
-rw------- 1 root root 5.0K Feb 14 19:27 common_voice_ur_26562739.pt
-rw------- 1 root root 4.3K Feb 14 19:27 common_voice_ur_26562740.pt
-rw------- 1 root root 4.3K Feb 14 19:27 common_voice_ur_26562741.pt


In [ ]:
%%bash
PRE="/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning/MyTTSDataset/preprocess"
echo "pt_count=$(ls "$PRE"/*.pt 2>/dev/null | wc -l)"
du -sh "$PRE"


pt_count=12509
61M	/content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning/MyTTSDataset/preprocess


In [ ]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

cd /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning

python train.py


2026-02-14 20:24:40 - ChatterboxFinetune - INFO - --- Starting Chatterbox Finetuning ---
2026-02-14 20:24:40 - ChatterboxFinetune - INFO - Mode: CHATTERBOX-TTS
All necessary models are available under 'pretrained_models'.
2026-02-14 20:24:40 - ChatterboxFinetune - INFO - Device: cuda
2026-02-14 20:24:40 - ChatterboxFinetune - INFO - Model Directory: ./pretrained_models
2026-02-14 20:24:40 - ChatterboxFinetune - INFO - Loading original model to extract weights...
2026-02-14 20:24:53 - ChatterboxFinetune - INFO - Creating new T3 model with vocab size: 2454
2026-02-14 20:25:00 - ChatterboxFinetune - INFO - Transferring weights...
2026-02-14 20:25:01 - src.model - INFO - Embedding layer: 704 tokens preserved.
2026-02-14 20:25:01 - src.model - INFO - Embedding layer: 1750 new tokens initialized with mean.
2026-02-14 20:25:01 - src.model - INFO - Output head: 704 tokens preserved.
2026-02-14 20:25:01 - src.model - INFO - Output head: 1750 new neurons initialized with mean.
2026-02-14 20:25:0

/root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
INFO:ChatterboxFinetune:Creating new T3 model with vocab size: 2454
INFO:ChatterboxFinetune:Transf

In [ ]:
%%bash
set -e
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

# Go to chatterbox-TTS repo
cd /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-TTS

# Install it in editable mode (gives s3tokenizer + chatterbox deps)
pip install -e .

echo "✅ chatterbox-TTS installed successfully"


Obtaining file:///content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-TTS
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 108.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 73.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 120.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 93.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 35.0 MB/s  0:00:15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.16.2 requires torch==2.1.2, but you have torch 2.6.0 which is incompatible.


##  infrencin on model  

In [ ]:
%%bash
set -e

# Activate micromamba env
eval "$(./bin/micromamba shell hook -s bash)"
micromamba activate tts

# Go to repo
cd /content/drive/MyDrive/urdu_tts_clean/repos/chatterbox-finetuning

# Run inference
python inference.py


2026-02-16 12:09:09 - Chatterbox-Inference - INFO - Running inference on device: cuda
2026-02-16 12:09:09 - Chatterbox-Inference - INFO - Preparing reference audio...
2026-02-16 12:09:10 - Chatterbox-Inference - INFO - Clean reference saved at: /content/drive/MyDrive/urdu_tts_clean/repos/whatsapp_reference_clean.wav
2026-02-16 12:09:10 - Chatterbox-Inference - INFO - Loading model in NORMAL mode...
2026-02-16 12:09:10 - Chatterbox-Inference - INFO - Base model directory: ./pretrained_models
2026-02-16 12:09:23 - Chatterbox-Inference - INFO - Rebuilding new T3 with vocab size: 2454
2026-02-16 12:09:31 - Chatterbox-Inference - INFO - Loading finetuned weights from: ./chatterbox_output/t3_finetuned.safetensors
2026-02-16 12:09:33 - Chatterbox-Inference - INFO - Fine-tuned engine loaded successfully.
2026-02-16 12:09:33 - Chatterbox-Inference - INFO - Total sentences: 2
2026-02-16 12:09:33 - Chatterbox-Inference - INFO - Synthesizing (1/2): پاکستان ایک خوبصورت ملک ہے جہاں مختلف ثقافتیں آبا

/root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master
/root/.local/share/mamba/envs/tts/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
INFO:Chatterbox-Inference:R